# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GazalaNK/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

one row = one content page, over a month with its total impressions, clicks, and average search position.Table(s): fact_content_daily_performance (daily grain, aggregated up to page-per-month) joined to dim_content for content type/word count context.
Time window: one mid-panel month, month=2026-03 (not the _sample table, since that's the sealed final month — using it for label logic would leak the future).
What I'd predict/rank: a CTR residual — actual CTR minus the expected CTR for that page's position tier — ranked so the worst under-performers surface first.
What I deliberately exclude: any FlyRank product-computed fields (health_score, priority_score, action_type) — my data ships observable signals only, so there's nothing to strip, but I'm noting explicitly that I will never rebuild and reuse those as features.

In [13]:
from huggingface_hub import notebook_login
notebook_login()

In [14]:

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')  # you set this once in Colab's key icon panel
print(hf_token[:6])  # should print something like "hf_XXX" — just the prefix, not the whole token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

hf_Tmp


┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

q1 = con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print("Duplicate rows (should be empty):", q1.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows (should be empty): (0, 3)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position — raw observed signals, known before any decision.
Label/proxy: CTR residual (actual CTR − expected CTR for the page's position tier) — derived from features above, not a separate field.
Context: content_type, word_count (from dim_content) — useful for later grouping, not used as predictive features yet.
Excluded: health_score, priority_score, action_type, refresh_tier — FlyRank's own product decisions. Excluded because using them as features would mean the model just learns to copy FlyRank's existing rule instead of finding real signal (circular result).

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample = con.sql("""
    SELECT gsc_impressions, gsc_clicks, gsc_avg_position, gsc_data_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    LIMIT 5
""").df()
sample


,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available
0,20,0,3.350000,True
1,1,0,0.000000,True
2,125,1,4.928000,True
3,7,0,4.000000,True
4,11,0,2.272727,True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.# Claim: one row = one page per day (grain check)
q1 = con.sql("""
    SELECT content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()
print(q1.shape)  # expect (0, 3) — no duplicates


# Claim: this slice is March 2026, with a real row count
q2 = con.sql("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
print(q2)


# Claim: filtering to available rows only, using IS TRUE
q3 = con.sql("""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
    WHERE gsc_data_available IS TRUE
""").df()
print(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(0, 3)
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   available_rows
0         3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice covers only March 2026 — one month, so it can't show seasonality or longer trends. Clients have different tracking start dates (unbalanced panel), so early rows for some clients may only have search data with no GA4 engagement data yet. Position-tier expectations built from one month may not generalize to other months or to the full warehouse.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.